In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import FunctionTransformer
import joblib

In [29]:
dataframe = pd.read_csv("train.csv",low_memory=False)
dataframe["attack_cat"] = (
    dataframe["attack_cat"]
    .str.strip()
    .str.lower()
)
dataframe.shape

(220153, 49)

In [30]:
# dataframe = dataframe.sample(frac=0.2) #use it for testing only

In [31]:
# filling missing values and converting string numerics to numeric
dataframe["ct_flw_http_mthd"] = dataframe["ct_flw_http_mthd"].fillna(0)
dataframe["is_ftp_login"] = dataframe["is_ftp_login"].fillna(0)
dataframe["service"] = dataframe["service"].replace('-', 'None').fillna('None')

# Safely convert ports to numeric, replacing non-numeric with -1
for col in ["sport", "dport", "ct_ftp_cmd"]:
  try:
    dataframe[col] = pd.to_numeric(dataframe[col], errors='coerce').fillna(-1).astype(int)
  except:
    print("issue")

issue


In [32]:
#removing unnecessary columns
redundantCols = ["Label", "srcip","dstip","sttl","dttl","Stime","Ltime"]
for col in redundantCols:
    dataframe.drop(col,axis=1,inplace=True)
dataframe["attack_cat"] = dataframe["attack_cat"].replace("backdoors","backdoor")


In [33]:
#clubbing the labels into a class called others, commented because it was not a good practise to improve accuracy
dataframe["attack_cat"] = dataframe["attack_cat"].replace("analysis", "reconnaissance")
dataframe["attack_cat"] = dataframe["attack_cat"].replace("backdoor", "malware")
dataframe["attack_cat"] = dataframe["attack_cat"].replace("shellcode", "malware")
dataframe["attack_cat"] = dataframe["attack_cat"].replace("worms", "malware")
dataframe["attack_cat"] = dataframe["attack_cat"].replace("exploits", "exploit")
dataframe["attack_cat"] = dataframe["attack_cat"].replace("generic", "exploit")
dataframe["attack_cat"] = dataframe["attack_cat"].replace("fuzzers", "exploit")
dataframe = dataframe.dropna()

In [34]:
# creating test and dev set
split = StratifiedShuffleSplit(random_state=42,test_size=0.2,n_splits=10)
for trainIndex,testIndex in split.split(dataframe,dataframe["attack_cat"]):
    train = dataframe.iloc[trainIndex]
    test = dataframe.iloc[testIndex]

In [35]:
# seperating features and labels
X_train = train.drop(columns=["attack_cat"])
y_train = train["attack_cat"]
X_test = test.drop(columns=["attack_cat"])
y_test = test["attack_cat"]

In [36]:
# seperating numeric and categorical attributes
trainNumAttributes = X_train.select_dtypes(exclude="object").columns.tolist()
testNumAttributes = X_test.select_dtypes(exclude="object").columns.tolist()
trainCatAttributes = X_train.select_dtypes(include="object").columns.tolist()
testCatAttributes = X_test.select_dtypes(include="object").columns.tolist()

In [37]:
from sklearn.preprocessing import OneHotEncoder

# Identify low and high cardinality categorical columns
# 'proto' and 'service' often have many values
cat_cols = trainCatAttributes

numPipeline = Pipeline([
    ("scaler", StandardScaler())
])

catPipeline = Pipeline([
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

fullPipeline = ColumnTransformer([
    ("num", numPipeline, trainNumAttributes),
    ("cat", catPipeline, trainCatAttributes)
])

In [38]:
from sklearn.ensemble import RandomForestClassifier

# Using Balanced Random Forest logic via class_weight and increasing estimators for better stability
models = {



    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(
            n_neighbors=7,
            weights="distance",
            metric="manhattan",
            p=2,
            n_jobs=-1
        ))
    ]),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            class_weight="balanced",
            probability=True,
            random_state=42
        ))
    ]),
    "RandomForest": RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            multi_class="multinomial",
            solver="lbfgs",
            C=1.0,
            class_weight="balanced",
            random_state=42,
            max_iter=1000
        ))
    ])
}

In [39]:
X = fullPipeline.fit_transform(X_train)
X2 = fullPipeline.transform(X_test)
y_test.value_counts()

,count
attack_cat,
normal,38444
exploit,4952
reconnaissance,290
dos,275
malware,70


In [ ]:
for model in models:
  kFold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
  print(f"training {model}...")
  modelObject = models[model]


  for fold, (trainIdx, valIdx) in enumerate(kFold.split(X, y_train)):
    XFoldTrain, XFoldVal = X[trainIdx], X[valIdx]
    yFoldTrain, yFoldVal = y_train.iloc[trainIdx], y_train.iloc[valIdx]

    modelObject.fit(XFoldTrain, yFoldTrain)
    foldPreds = modelObject.predict(XFoldVal)
    acc = accuracy_score(yFoldVal, foldPreds)
    print(f"Fold {fold+1} Accuracy: {acc:.4f}")

  # Final evaluation on the test set X2
  predictions = modelObject.predict(X2)

  class_report = classification_report(
    y_test,
    predictions

  )
  print(f"\nClassification Report for {model}:")
  print(class_report)
  print("-" * 30)
  joblib.dump(modelObject, f"{model}.pkl")

training KNN...
Fold 1 Accuracy: 0.9776
Fold 2 Accuracy: 0.9777
Fold 3 Accuracy: 0.9791
Fold 4 Accuracy: 0.9778
Fold 5 Accuracy: 0.9775
Fold 6 Accuracy: 0.9789
Fold 7 Accuracy: 0.9777
Fold 8 Accuracy: 0.9775
Fold 9 Accuracy: 0.9777
Fold 10 Accuracy: 0.9784

Classification Report for KNN:
                precision    recall  f1-score   support

           dos       0.30      0.29      0.30       275
       exploit       0.91      0.91      0.91      4952
       malware       0.14      0.04      0.07        70
        normal       0.99      1.00      0.99     38444
reconnaissance       0.61      0.52      0.56       290

      accuracy                           0.98     44031
     macro avg       0.59      0.55      0.56     44031
  weighted avg       0.98      0.98      0.98     44031

------------------------------
training SVM...
